<a href="https://colab.research.google.com/github/orgernn/Roles-classification-using-Sentence-Transformers/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install sentence-transformers

In [9]:
from sentence_transformers import SentenceTransformer, util
import numpy as np

In [10]:
# הגדרת המודל שבאמצעותו נפיק מהמילים מערכים עם ערך עשרוני בהתאם למשמעות שלהן
#מכיוון שהוא מאוד קל ומהיר all-MiniLM-L6-v2 בחרתי להשתמש במודל
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [11]:
#כל תפקידי הרע"ן בחטיבה האסטרטגית, מסודרים ברשימת מערכים שבהם ומילות קוד המסמנות את תחומי העיסוק שלהם
roles = {
  "north": ["labanon", "Syria", "Damascus ", "hezbollah", "Civil War", "Buffer Zone", "Unmanned Aerial Vehicles", "Beirut", "UNIFIL", "Lebanese Armed Forces"],
  "middle east": ["egypt", "jordan", "Alliances", "yemen", "iraq", "qatar", "oman", "kuwait", "azerbaijan"],
  "iran": ["Islamic Revolutionary Guard Corps", "iran", "Nuclear weapon", "Axis of Resistance", "Proxies", "Houthis", "Tehran"],
  "palestinian": ["Gaza Strip", "Hamas", "Rockets", "Hostages", "Humanitarian Aid", "Palestinian Authority", "Settlements", "Border Police"],
  "world": ["United States", "NATO", "United Kingdom", "France", "Russia", "Germany", "Indo-Pacific", "Geopolitics", "Defense Cooperation", "Taiwan"]
}
role_names = list(roles.keys())

# נמיר את המערכים ברשימת התפקידים לערכים עשרוניים
role_embeddings = []
for name in role_names:
  keyword_embs = model.encode(roles[name])
  centroid = np.mean(keyword_embs, axis=0)
  role_embeddings.append(centroid)
role_embeddings = np.array(role_embeddings)
# print(role_embeddings.shape)

# הגדרת הכותרת שעליה נבצע את הקלסיפיקציה
article_title = "Crowds fill Tehran's streets for slain ayatollah's funeral procession"

# אם נרצה לקבל מהמשתמש את הכותרת
# article_title = input("Enter the article title: ")

# המרת הכותרת לערך עשרוני
article_embedding = model.encode(article_title)

# הפקת התאמה בין הכותרת לבין הקטגוריות באמצעות פונקציית קוסיין, המחזירה ערך בין -1 ל1, כאשר -1 מייצג הפכים גמורים, 0 מייצג אי קשר מוחלט ו1 התאמה מושלמת
scores = util.cos_sim(article_embedding, role_embeddings)[0]
best_role = role_names[scores.argmax()]

# הצגת התוצאות בצורה ברורה, וחישוב אחוזי התאמה
results = sorted(zip(role_names, scores.tolist()), key=lambda x: x[1], reverse=True)
print("best role:" + best_role)

for name, score in results:
  print(f"{name:15s}: {round(score*100, 2)}%")


best role: iran
iran           : 30.22%
middle east    : 22.74%
north          : 19.28%
palestinian    : 18.04%
world          : 8.47%


In [21]:
import json
from google.colab import drive

# Mount your drive if the file is there, or just define the filename
# If your notebook is saved as "MyNotebook.ipynb"
filename = "/content/drive/MyDrive/Colab Notebooks/main.ipynb"

with open(filename, 'r', encoding='utf-8') as f:
    nb = json.load(f)

# Remove the 'widgets' key from metadata if it exists
if 'metadata' in nb and 'widgets' in nb['metadata']:
    del nb['metadata']['widgets']
    print("Widgets metadata removed.")
else:
    print("No widget metadata found.")

with open(filename, 'w', encoding='utf-8') as f:
    json.dump(nb, f)
    print("File saved. Try pushing to GitHub again.")

Widgets metadata removed.
File saved. Try pushing to GitHub again.
